In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import json
import uuid
from confluent_kafka import Consumer, KafkaError

def read_kafka_topic_to_df(topic: str, bootstrap_servers: str, timeout_sec: float = 5.0) -> pd.DataFrame:
    """
    Вичитує всі доступні повідомлення з Kafka топіку в pandas DataFrame 
    без збереження оффсетів (не відмічаючи як прочитані).
    """
    conf = {
        'bootstrap.servers': bootstrap_servers,
        'group.id': f'pandas_analyzer_{uuid.uuid4()}', 
        'auto.offset.reset': 'earliest',
        'enable.auto.commit': False,
    }

    consumer = Consumer(conf)
    
    try:
        consumer.subscribe([topic])
        print(f"Підписано на топік: {topic}. Починаю вичитування...")
        
        data = []
        
        while True:
            msg = consumer.poll(timeout=timeout_sec)
            
            if msg is None:
                print(f"Немає нових повідомлень протягом {timeout_sec} сек. Завершення читання.")
                break
                
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    continue
                else:
                    print(f"Помилка Kafka: {msg.error()}")
                    break

            try:
                value_str = msg.value().decode('utf-8')
                value_dict = json.loads(value_str)
                data.append(value_dict)
            except Exception as e:
                print(f"Помилка декодування: {e}")

    finally:
        consumer.close()
        print(f"Консьюмер закрито. Всього вичитано повідомлень: {len(data)}")

    return pd.DataFrame(data)

In [3]:
bootstrap_servers = 'localhost:19092'
topic = 'news-parsed'

df = read_kafka_topic_to_df(bootstrap_servers=bootstrap_servers, topic=topic)
df.head()

Підписано на топік: news-parsed. Починаю вичитування...
Немає нових повідомлень протягом 5.0 сек. Завершення читання.
Консьюмер закрито. Всього вичитано повідомлень: 5449


,title,description,link,image_url,published_at,source_id
0,росіяни вночі вдарили по Дніпру балістикою є п...,росіяни вночі вдарили по Дніпру балістикою є п...,https://unn.ua/news/rosiiany-vnochi-vdaryly-po...,https://unn.ua/img/2026/04/16/1776300112-1832-...,2026-04-16T00:41:56Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1
1,"Інфантіно заявив, що Іран візьме участь у чемп...","Інфантіно заявив, що Іран візьме участь у чемп...",https://unn.ua/news/infantino-zaiavyv-shcho-ir...,https://unn.ua/img/2026/04/15/1776293154-1132-...,2026-04-15T22:45:57Z,8fbd6207-4ebc-4b96-93c8-8197ee3925e1
2,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,https://unn.ua/news/ataky-rf-na-dnipro-15-kvit...,https://unn.ua/img/2026/04/15/1776252519-2000-...,2026-04-15T11:28:42Z,9469aa98-da2f-4b40-b98d-db991ca85f51
3,Amazon представила авіаційну антену супутников...,Антена на літаку дає доступ до альтернативи St...,https://ua.korrespondent.net/tech/technews/487...,https://kor.ill.in.ua/m/190x120/4402706.jpg,2026-04-14T12:57:00Z,1f37bd0e-7185-48e2-8124-718a201e98c9
4,"""Українська бронетехніка"" модернізує ударні др...","""Українська бронетехніка"" модернізує ударні др...",https://unn.ua/news/ukrainska-bronetekhnika-mo...,https://unn.ua/img/2026/04/15/1776292651-3940-...,2026-04-15T22:37:33Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5449 entries, 0 to 5448
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         5449 non-null   str  
 1   description   5449 non-null   str  
 2   link          5449 non-null   str  
 3   image_url     4339 non-null   str  
 4   published_at  5444 non-null   str  
 5   source_id     5449 non-null   str  
dtypes: str(6)
memory usage: 255.6 KB


In [5]:
import os
from dotenv import load_dotenv

env_path = os.path.abspath('../.env')
load_dotenv(dotenv_path=env_path)

ADMIN_TOKEN = os.getenv('ADMIN_TOKEN')
API_URL = os.getenv('API_URL')
API_URL

'http://localhost:5000/api/v1'

In [6]:
categories = ['Війна', 'Політика', 'Економіка', 'Технології', 'Наука', 'Освіта', 'Спорт', 'Культура', 'Здоров\'я', 'Туризм']
categories

['Війна',
 'Політика',
 'Економіка',
 'Технології',
 'Наука',
 'Освіта',
 'Спорт',
 'Культура',
 "Здоров'я",
 'Туризм']

In [7]:
import requests

headers = {
    'Authorization': f'Bearer {ADMIN_TOKEN}',
}

category_mapping = {}

for category in categories:
    r = requests.get(
        f'{API_URL}/sources',
        params={
            'pageSize': 1000,
            'search': category,
        },
        headers=headers,
    )
    res = r.json()
    for item in res['data']:
        if item['id'] not in category_mapping:
            category_mapping[item['id']] = category
    print(f'Категорія {category} оброблена: {res['totalCount']} нових джерел')

print(f'Розмір словника джерел: {len(category_mapping)}')

Категорія Війна оброблена: 9 нових джерел
Категорія Політика оброблена: 11 нових джерел
Категорія Економіка оброблена: 10 нових джерел
Категорія Технології оброблена: 8 нових джерел
Категорія Наука оброблена: 4 нових джерел
Категорія Освіта оброблена: 3 нових джерел
Категорія Спорт оброблена: 15 нових джерел
Категорія Культура оброблена: 6 нових джерел
Категорія Здоров'я оброблена: 6 нових джерел
Категорія Туризм оброблена: 4 нових джерел
Розмір словника джерел: 76


In [8]:
df['category'] = df['source_id'].map(category_mapping)
df.head()

,title,description,link,image_url,published_at,source_id,category
0,росіяни вночі вдарили по Дніпру балістикою є п...,росіяни вночі вдарили по Дніпру балістикою є п...,https://unn.ua/news/rosiiany-vnochi-vdaryly-po...,https://unn.ua/img/2026/04/16/1776300112-1832-...,2026-04-16T00:41:56Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1,Війна
1,"Інфантіно заявив, що Іран візьме участь у чемп...","Інфантіно заявив, що Іран візьме участь у чемп...",https://unn.ua/news/infantino-zaiavyv-shcho-ir...,https://unn.ua/img/2026/04/15/1776293154-1132-...,2026-04-15T22:45:57Z,8fbd6207-4ebc-4b96-93c8-8197ee3925e1,Спорт
2,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,https://unn.ua/news/ataky-rf-na-dnipro-15-kvit...,https://unn.ua/img/2026/04/15/1776252519-2000-...,2026-04-15T11:28:42Z,9469aa98-da2f-4b40-b98d-db991ca85f51,Освіта
3,Amazon представила авіаційну антену супутников...,Антена на літаку дає доступ до альтернативи St...,https://ua.korrespondent.net/tech/technews/487...,https://kor.ill.in.ua/m/190x120/4402706.jpg,2026-04-14T12:57:00Z,1f37bd0e-7185-48e2-8124-718a201e98c9,Технології
4,"""Українська бронетехніка"" модернізує ударні др...","""Українська бронетехніка"" модернізує ударні др...",https://unn.ua/news/ukrainska-bronetekhnika-mo...,https://unn.ua/img/2026/04/15/1776292651-3940-...,2026-04-15T22:37:33Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1,Війна


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5449 entries, 0 to 5448
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         5449 non-null   str  
 1   description   5449 non-null   str  
 2   link          5449 non-null   str  
 3   image_url     4339 non-null   str  
 4   published_at  5444 non-null   str  
 5   source_id     5449 non-null   str  
 6   category      3915 non-null   str  
dtypes: str(7)
memory usage: 298.1 KB


In [10]:
df.category.value_counts()

category
Війна         765
Політика      744
Економіка     666
Спорт         564
Технології    300
Культура      294
Здоров'я      267
Наука         119
Освіта        115
Туризм         81
Name: count, dtype: int64

In [11]:
df[df.description.str.contains('<')].groupby('source_id').first().description.to_numpy()

array(['Бразильський президент Лула підтримав Папу Лева після критики Трампа<p>Президент Бразилії висловив солідарність понтифіку через його заклики до миру. Трамп назвав Папу слабким після слів про\nнеприпустимість ударів по Ірану.</p>',
       'Рада розгляне ініціативу щодо зміни порядку та строків сплати штрафів, зафіксованих камерами автофіксації<p>Законопроєкт № 13614 пропонує збільшити термін добровільної оплати до 15 днів. Час на передачу постанови до примусового виконання\nзросте до 3 місяців.</p>',
       'США розширили санкції проти мережі іранського нафтового магната, щоб тиснути на Тегеран<p>Вашингтон запровадив обмеження проти компаній та суден за тіньовий експорт нафти. Санкції стосуються схем відмивання коштів через\nзолото Венесуели.</p>',
       "В Україні готують державну політику з промоції читання - розпочинають з національного дослідження<p>Міністерство культури вивчить мотивацію та бар'єри українців у читанні для формування державної політики. Результати оприлюдня

In [12]:
import sys

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from analyzer.preprocessing import html_sanitize

html_sanitize

<function analyzer.preprocessing.html_sanitize(s: str) -> str>

In [13]:
df['title'] = df['title'].map(html_sanitize)
df['description'] = df['description'].map(html_sanitize)
df.head()

,title,description,link,image_url,published_at,source_id,category
0,росіяни вночі вдарили по Дніпру балістикою є п...,росіяни вночі вдарили по Дніпру балістикою є п...,https://unn.ua/news/rosiiany-vnochi-vdaryly-po...,https://unn.ua/img/2026/04/16/1776300112-1832-...,2026-04-16T00:41:56Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1,Війна
1,"Інфантіно заявив, що Іран візьме участь у чемп...","Інфантіно заявив, що Іран візьме участь у чемп...",https://unn.ua/news/infantino-zaiavyv-shcho-ir...,https://unn.ua/img/2026/04/15/1776293154-1132-...,2026-04-15T22:45:57Z,8fbd6207-4ebc-4b96-93c8-8197ee3925e1,Спорт
2,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,https://unn.ua/news/ataky-rf-na-dnipro-15-kvit...,https://unn.ua/img/2026/04/15/1776252519-2000-...,2026-04-15T11:28:42Z,9469aa98-da2f-4b40-b98d-db991ca85f51,Освіта
3,Amazon представила авіаційну антену супутников...,Антена на літаку дає доступ до альтернативи St...,https://ua.korrespondent.net/tech/technews/487...,https://kor.ill.in.ua/m/190x120/4402706.jpg,2026-04-14T12:57:00Z,1f37bd0e-7185-48e2-8124-718a201e98c9,Технології
4,"""Українська бронетехніка"" модернізує ударні др...","""Українська бронетехніка"" модернізує ударні др...",https://unn.ua/news/ukrainska-bronetekhnika-mo...,https://unn.ua/img/2026/04/15/1776292651-3940-...,2026-04-15T22:37:33Z,6cdf1e5f-0fc7-48b1-a797-126f96b330d1,Війна


In [14]:
df[df.description.str.contains('<')]['source_id'].value_counts()

Series([], Name: count, dtype: int64)

In [15]:
df[df.description.str.contains('<')].description.to_numpy()

array([], dtype=object)

In [16]:
df.description[0], df.title[0]

("росіяни вночі вдарили по Дніпру балістикою є поранені та руйнування. росіяни поцілили у житлові квартали Дніпра, пошкодивши багатоповерхівку та приватний будинок. Постраждали п'ятеро людей, одна жінка у важкому стані.",
 'росіяни вночі вдарили по Дніпру балістикою є поранені та руйнування')

In [17]:
# 1. Підготовка даних
# Відкидаємо рядки, де категорія NaN
ml_df = df.dropna(subset=['category']).copy()

# Об'єднуємо заголовок та опис для кращого контексту
ml_df['text'] = ml_df['title'] + " " + ml_df['description']

print(f"Кількість новин для тренування: {len(ml_df)}")
ml_df[['text', 'category']].head()

Кількість новин для тренування: 3915


,text,category
0,росіяни вночі вдарили по Дніпру балістикою є п...,Війна
1,"Інфантіно заявив, що Іран візьме участь у чемп...",Спорт
2,Атаки рф на Дніпро 15 квітня - пошкодежно три ...,Освіта
3,Amazon представила авіаційну антену супутников...,Технології
4,"""Українська бронетехніка"" модернізує ударні др...",Війна


In [18]:
# 2. Стратифіковане розбиття вибірки
from sklearn.model_selection import train_test_split

X = ml_df['text']
y = ml_df['category']

# Використовуємо stratify=y для збереження пропорцій незбалансованих класів
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Тренувальна вибірка: {len(X_train)}")
print(f"Тестова вибірка: {len(X_test)}")

Тренувальна вибірка: 3132
Тестова вибірка: 783


In [19]:
# 3. Створення та навчання Pipeline (TF-IDF + LinearSVC)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# Пайплайн автоматично векторизує текст і подає на вхід класифікатору
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2))),
    ('clf', LinearSVC(random_state=42))
])

text_clf.fit(X_train, y_train)
print("Модель успішно натренована!")

Модель успішно натренована!


In [20]:
# 4. Оцінка якості
from sklearn.metrics import classification_report, accuracy_score

predictions = text_clf.predict(X_test)

print(f"Загальна точність (Accuracy): {accuracy_score(y_test, predictions):.3f}\n")
print(classification_report(y_test, predictions))

Загальна точність (Accuracy): 0.688

              precision    recall  f1-score   support

       Війна       0.70      0.70      0.70       153
   Економіка       0.64      0.74      0.68       133
    Здоров'я       0.69      0.68      0.69        53
    Культура       0.83      0.64      0.72        59
       Наука       0.44      0.46      0.45        24
      Освіта       0.86      0.52      0.65        23
    Політика       0.61      0.74      0.67       149
       Спорт       0.92      0.85      0.88       113
  Технології       0.51      0.43      0.47        60
      Туризм       1.00      0.31      0.48        16

    accuracy                           0.69       783
   macro avg       0.72      0.61      0.64       783
weighted avg       0.70      0.69      0.69       783



In [21]:
# 5. Збереження моделі у файл
import joblib

# Зберігаємо у кореневу папку проєкту для використання в сервісі
model_path = os.path.join(PROJECT_ROOT, 'news_categorization_model.joblib')
joblib.dump(text_clf, model_path)

print(f"Модель збережено за шляхом: {model_path}")

Модель збережено за шляхом: /home/andrii/University/news-aggregator/news-analyzer/news_categorization_model.joblib


## Дослідження найкращої конфігурації (Grid Search)

Ми використаємо `GridSearchCV`, щоб перевірити різні комбінації параметрів:
1. **TF-IDF**: розмір біграм, максимальна частота слів.
2. **LinearSVC**: параметр регуляризації `C` та метод обробки незбалансованих класів.

In [22]:
from sklearn.model_selection import GridSearchCV

# Визначаємо сітку параметрів
param_grid = {
    'tfidf__max_df': [0.8, 0.9, 1.0],
    'tfidf__ngram_range': [(1, 1), (1, 2)], # тільки слова або слова + біграми
    'clf__C': [0.1, 1, 10, 100],
    'clf__class_weight': [None, 'balanced'] # балансування ваг для рідкісних категорій
}

# Створюємо GridSearchCV
# cv=5: 5-кратна крос-валідація
# scoring='f1_weighted': метрика, що найкраще підходить для незбалансованих даних
grid_search = GridSearchCV(
    text_clf, 
    param_grid, 
    cv=5, 
    n_jobs=-1, 
    scoring='f1_weighted', 
    verbose=2
)

print("Починаю пошук оптимальних параметрів... (це може зайняти час)")
grid_search.fit(X_train, y_train)

print(f"\nНайкращий F1-weighted score: {grid_search.best_score_:.4f}")
print("Найкращі параметри:", grid_search.best_params_)

Починаю пошук оптимальних параметрів... (це може зайняти час)
Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   0.1s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   0.1s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.1s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.1s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   0.3s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.8, tfidf__ngram_range=(1, 2); total time=   0.2s
[CV] END clf__C=0.1, clf__class_weight=None, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   0

/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END clf__C=10, clf__class_weight=None, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   0.8s
[CV] END clf__C=10, clf__class_weight=None, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   0.9s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.8, tfidf__ngram_range=(1, 1); total time=   1.0s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.6s


/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.9, tfidf__ngram_range=(1, 1); total time=   0.6s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.8, tfidf__ngram_range=(1, 2); total time=   1.2s
[CV] END clf__C=10, clf__class_weight=balanced, tfidf__max_df=0.8, tfidf__ngram_range=(1, 2); total time=   1.0s


...

/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END clf__C=100, clf__class_weight=balanced, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   1.4s


/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END clf__C=100, clf__class_weight=balanced, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   2.0s
[CV] END clf__C=100, clf__class_weight=balanced, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   1.4s
[CV] END clf__C=100, clf__class_weight=balanced, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   1.4s
[CV] END clf__C=100, clf__class_weight=balanced, tfidf__max_df=1.0, tfidf__ngram_range=(1, 2); total time=   1.3s

Найкращий F1-weighted score: 0.7181
Найкращі параметри: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'tfidf__max_df': 0.8, 'tfidf__ngram_range': (1, 2)}


/home/andrii/University/news-aggregator/news-analyzer/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [23]:
# Оцінка найкращої моделі на тестовій вибірці
best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)

print("Звіт по найкращій моделі з Grid Search:\n")
print(classification_report(y_test, best_predictions))

# Збереження найкращої моделі
final_model_path = os.path.join(PROJECT_ROOT, 'news_categorization_model_optimized.joblib')
joblib.dump(best_model, final_model_path)
print(f"Оптимізовану модель збережено: {final_model_path}")

Звіт по найкращій моделі з Grid Search:

              precision    recall  f1-score   support

       Війна       0.74      0.69      0.71       153
   Економіка       0.65      0.65      0.65       133
    Здоров'я       0.63      0.83      0.72        53
    Культура       0.69      0.68      0.68        59
       Наука       0.34      0.50      0.41        24
      Освіта       0.62      0.65      0.64        23
    Політика       0.67      0.72      0.69       149
       Спорт       0.92      0.87      0.89       113
  Технології       0.55      0.38      0.45        60
      Туризм       0.55      0.38      0.44        16

    accuracy                           0.69       783
   macro avg       0.64      0.63      0.63       783
weighted avg       0.69      0.69      0.69       783

Оптимізовану модель збережено: /home/andrii/University/news-aggregator/news-analyzer/news_categorization_model_optimized.joblib


## Дослідження найкращої конфігурації для Random Forest

Тепер ми спробуємо використати `RandomForestClassifier` і порівняємо його результати з `LinearSVC`.

In [24]:
from sklearn.ensemble import RandomForestClassifier

# Створення нового пайплайну для Random Forest
rf_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.9, min_df=2)),
    ('clf', RandomForestClassifier(random_state=42))
])

# Визначаємо сітку параметрів для Random Forest
rf_param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__n_estimators': [100, 300, 500],
    'clf__max_depth': [None, 20, 50],
    'clf__class_weight': [None, 'balanced']
}

# GridSearchCV для Random Forest
rf_grid_search = GridSearchCV(
    rf_clf, 
    rf_param_grid, 
    cv=5, 
    n_jobs=-1, 
    scoring='f1_weighted', 
    verbose=2
)

print("Починаю пошук оптимальних параметрів для Random Forest...")
rf_grid_search.fit(X_train, y_train)

print(f"\nНайкращий F1-weighted score (RF): {rf_grid_search.best_score_:.4f}")
print("Найкращі параметри (RF):", rf_grid_search.best_params_)

Починаю пошук оптимальних параметрів для Random Forest...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 1); total time=   6.4s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 1); total time=   6.8s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 1); total time=   7.2s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 1); total time=   7.7s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 1); total time=   7.8s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 2); total time=   9.4s
[CV] END clf__class_weight=None, clf__max_depth=None, clf__n_estimators=100, tfidf__ngram_range=(1, 2); total time=   9.9s
[CV] END clf__class

In [25]:
# Оцінка найкращої моделі Random Forest
best_rf_model = rf_grid_search.best_estimator_
rf_predictions = best_rf_model.predict(X_test)

print("Звіт по найкращій моделі Random Forest:\n")
print(classification_report(y_test, rf_predictions))

# Збереження моделі
rf_model_path = os.path.join(PROJECT_ROOT, 'news_categorization_rf_model.joblib')
joblib.dump(best_rf_model, rf_model_path)
print(f"Модель Random Forest збережено: {rf_model_path}")

Звіт по найкращій моделі Random Forest:

              precision    recall  f1-score   support

       Війна       0.62      0.68      0.65       153
   Економіка       0.53      0.60      0.56       133
    Здоров'я       0.64      0.60      0.62        53
    Культура       0.83      0.41      0.55        59
       Наука       0.42      0.42      0.42        24
      Освіта       0.73      0.35      0.47        23
    Політика       0.57      0.67      0.62       149
       Спорт       0.85      0.86      0.85       113
  Технології       0.45      0.42      0.43        60
      Туризм       1.00      0.31      0.48        16

    accuracy                           0.62       783
   macro avg       0.66      0.53      0.56       783
weighted avg       0.64      0.62      0.62       783

Модель Random Forest збережено: /home/andrii/University/news-aggregator/news-analyzer/news_categorization_rf_model.joblib
